In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Caminho dos dados
RAW = "C:/Users/felip/OneDrive/Documentos/olist/data/raw/"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
orders       = pd.read_csv(RAW + "olist_orders_dataset.csv")
customers    = pd.read_csv(RAW + "olist_customers_dataset.csv")
items        = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments     = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews      = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
products     = pd.read_csv(RAW + "olist_products_dataset.csv")
sellers      = pd.read_csv(RAW + "olist_sellers_dataset.csv")
geolocation  = pd.read_csv(RAW + "olist_geolocation_dataset.csv")
translation  = pd.read_csv(RAW + "product_category_name_translation.csv")

print("Tabelas carregadas!")
print(f"Orders:      {orders.shape}")
print(f"Customers:   {customers.shape}")
print(f"Items:       {items.shape}")
print(f"Payments:    {payments.shape}")
print(f"Reviews:     {reviews.shape}")
print(f"Products:    {products.shape}")
print(f"Sellers:     {sellers.shape}")
print(f"Geolocation: {geolocation.shape}")
print(f"Translation: {translation.shape}")

Tabelas carregadas!
Orders:      (99441, 8)
Customers:   (99441, 5)
Items:       (112650, 7)
Payments:    (103886, 5)
Reviews:     (99224, 7)
Products:    (32951, 9)
Sellers:     (3095, 4)
Geolocation: (1000163, 5)
Translation: (71, 2)


In [3]:
# Converter datas
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("=== VISÃO GERAL DOS PEDIDOS ===")
print(f"\nTotal de pedidos: {len(orders):,}")
print(f"Período: {orders['order_purchase_timestamp'].min().date()} até {orders['order_purchase_timestamp'].max().date()}")
print(f"\nStatus dos pedidos:")
print(orders["order_status"].value_counts())

=== VISÃO GERAL DOS PEDIDOS ===

Total de pedidos: 99,441
Período: 2016-09-04 até 2018-10-17

Status dos pedidos:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [4]:
# Juntar pedidos com pagamentos
df = orders.merge(payments, on="order_id", how="left")
df = df.merge(customers, on="customer_id", how="left")

# Filtrar só pedidos entregues
df_entregues = df[df["order_status"] == "delivered"]

receita_total = df_entregues["payment_value"].sum()
ticket_medio  = df_entregues.groupby("order_id")["payment_value"].sum().mean()
total_clientes = df_entregues["customer_unique_id"].nunique()

print("=== MÉTRICAS PRINCIPAIS ===")
print(f"\nReceita total:    R$ {receita_total:,.2f}")
print(f"Ticket médio:     R$ {ticket_medio:,.2f}")
print(f"Pedidos entregues: {len(df_entregues['order_id'].unique()):,}")
print(f"Clientes únicos:  {total_clientes:,}")

=== MÉTRICAS PRINCIPAIS ===

Receita total:    R$ 15,422,461.77
Ticket médio:     R$ 159.85
Pedidos entregues: 96,478
Clientes únicos:  93,358


In [5]:
df_entregues = df_entregues.copy()
df_entregues["mes"] = df_entregues["order_purchase_timestamp"].dt.to_period("M").astype(str)

receita_mensal = df_entregues.groupby("mes")["payment_value"].sum().reset_index()
receita_mensal.columns = ["mes", "receita"]

fig = px.line(
    receita_mensal,
    x="mes", y="receita",
    title="Receita Mensal — Olist (2016-2018)",
    labels={"mes": "Mês", "receita": "Receita (R$)"},
    markers=True,
    height=450,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

print(f"\nMelhor mês: {receita_mensal.loc[receita_mensal['receita'].idxmax(), 'mes']} — R$ {receita_mensal['receita'].max():,.2f}")
print(f"Pior mês:   {receita_mensal.loc[receita_mensal['receita'].idxmin(), 'mes']} — R$ {receita_mensal['receita'].min():,.2f}")


Melhor mês: 2017-11 — R$ 1,153,528.05
Pior mês:   2016-09 — R$ 0.00


In [7]:
# Juntar items com products e translation
df_produtos = items.merge(products, on="product_id", how="left")
df_produtos = df_produtos.merge(translation, on="product_category_name", how="left")
df_produtos = df_produtos.merge(
    orders[["order_id", "order_status"]], on="order_id", how="left"
)

# Só pedidos entregues
df_produtos = df_produtos[df_produtos["order_status"] == "delivered"]

# Receita por categoria
receita_cat = df_produtos.groupby("product_category_name_english")["price"].sum()\
    .sort_values(ascending=False).head(15).reset_index()
receita_cat.columns = ["categoria", "receita"]

fig = px.bar(
    receita_cat,
    x="receita", y="categoria",
    orientation="h",
    title="Top 15 Categorias por Receita",
    labels={"receita": "Receita (R$)", "categoria": ""},
    height=500,
    color="receita",
    color_continuous_scale="Blues",
)
fig.update_layout(yaxis=dict(autorange="reversed"), showlegend=False)
fig.show()